In [0]:
from pyspark.sql.functions import current_timestamp, col
df_transactions = (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load("/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv")
)
df_transactions =(
     df_transactions
        .withColumn("_ingest_timestamp", current_timestamp()) 
        .withColumn("_source_file_name", col("_metadata.file_path"))
)
(
    df_transactions.write
    .option("overwriteSchema", "true")
    .format("delta")
    .mode("overwrite")
    .saveAsTable("databricks_project.schema.bronze_transactions")
    )
    

In [0]:
%sql
select * from databricks_project.schema.bronze_transactions

order_id,item_id,customer_id,quantity,price,order_timestamp,corrupted_flag,_ingest_timestamp,_source_file_name
ORD-101,PROD-001,CUST001,2.0,$49.99,2025-01-01T22:30:15.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-102,PROD-002,CUST002,3.0,$19.99,2025-01-02T18:15:00.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-103,PROD-003,CUST003,5.0,$9.99,2025-01-04T02:45:30.000Z,Y,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-104,PROD-004,CUST004,2.0,$299.99,2025-01-04T20:00:00.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-105,PROD-005,CUST005,1.0,$15.50,2025-01-05T17:20:10.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-106,PROD-006,CUST006,2.0,$75.00,2025-01-07T00:40:25.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-107,PROD-007,CUST007,4.0,$25.99,2025-01-07T19:10:45.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-108,PROD-008,CUST008,1.0,$89.99,2025-01-08T21:55:05.000Z,Y,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-109,PROD-009,CUST009,2.0,$45.00,2025-01-10T01:25:30.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv
ORD-110,PROD-010,CUST010,1.0,$120.00,2025-01-10T16:05:15.000Z,N,2026-04-09T07:25:36.861Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/transactions_raw.csv


In [0]:
df_products = (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load("/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv")
)
df_products = df_products.withColumns({
    "_ingest_timestamp": current_timestamp(),
    "_source_file_name" : col("_metadata.file_path")
})
# df_products.printSchema()
(
    df_products.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("databricks_project.schema.bronze_products")
)

In [0]:
%sql
select * from databricks_project.schema.bronze_products

item_id,product_name,category,_ingest_timestamp,_source_file_name
PROD-001,Laptop,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-002,Smartphone,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-003,Headphones,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-004,Tablet,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-005,Keyboard,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-006,Monitor,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-007,Mouse,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-008,Webcam,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-009,Speaker,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-010,Printer,ELECTRONICS,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, current_timestamp
df_customers = (
    spark.read.format("json")
    .load("/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json")
)

df_customers = df_customers.withColumns({
    "_ingest_timestamp" : current_timestamp(),
    "_source_file_name" : col("_metadata.file_path")
})
df_customers.display()

# window = Window.partitionBy("customer_id").orderBy(col("region").desc())

# df_customers = df_customers \
#     .withColumn("rn", row_number().over(window)) \
#     .filter(col("rn") == 1) \
#     .drop("rn")

(
    df_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databricks_project.schema.bronze_customers")
)


_corrupt_record,contact,customer_id,name,region,_ingest_timestamp,_source_file_name
null,List(john@example.com),CUST001,John Doe,West,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(jane@example.com),CUST002,Jane Smith,West,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(mike@example.com),CUST003,Mike Ross,South,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(rachel@example.com),CUST004,Rachel Zane,West,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(harvey@example.com),CUST005,Harvey Specter,East,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(donna@example.com),CUST006,Donna Paulsen,North,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(k@gmail.com),CUST007,Karthika,south,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(g@gmail.com),CUST008,guest1,North,2026-04-09T07:25:45.125Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json


In [0]:
%sql
select * from databricks_project.schema.bronze_customers

_corrupt_record,contact,customer_id,name,region,_ingest_timestamp,_source_file_name
null,List(john@example.com),CUST001,John Doe,West,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(jane@example.com),CUST002,Jane Smith,West,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(mike@example.com),CUST003,Mike Ross,South,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(rachel@example.com),CUST004,Rachel Zane,West,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(harvey@example.com),CUST005,Harvey Specter,East,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(donna@example.com),CUST006,Donna Paulsen,North,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(k@gmail.com),CUST007,Karthika,south,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
null,List(g@gmail.com),CUST008,guest1,North,2026-04-09T07:25:46.180Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/customers_raw.json
